# Flow-GRPO 代码实现详解

> **论文**: *Flow-GRPO: Group Relative Policy Optimization for Flow Matching Models*  
> **核心思想**: 将 LLM 领域的 GRPO 适配到连续时间 Flow Matching 模型（如 SD3、FLUX），用强化学习优化文本到图像生成。

## 本 Demo 覆盖的 12 个核心要点

| # | 要点 | 说明 |
|---|------|------|
| 1 | **Flow Matching 基础** | 速度场 $v_\theta$、ODE Euler 步进 |
| 2 | **ODE → SDE 转换** | Anderson 反向 SDE，注入噪声使采样有随机性 |
| 3 | **Log 概率计算** | 高斯转移核的 closed-form $\log p(x_{t-1}|x_t)$ |
| 4 | **CPS** | Coefficients-Preserving Sampling，端点预测重参数化 |
| 5 | **GRPO 采样** | 同一 prompt 生成 G 张图，形成 group |
| 6 | **Per-Prompt 优势** | $(reward - mean) / std$，无需 value function |
| 7 | **PPO 裁剪损失** | Clipped Surrogate Loss |
| 8 | **KL 正则化** | 当前策略与 reference model 的 KL |
| 9 | **Flow-GRPO-Fast** | 窗口化训练，仅部分时间步注入 SDE |
| 10 | **GRPO-Guard** | 比率偏差校正，防止过度优化 |
| 11 | **EMA** | 指数移动平均，用于评估和保存 |
| 12 | **LoRA** | 参数高效微调 |

下面逐模块拆解代码实现 👇

## 导入依赖与全局配置

所有超参数集中管理，涵盖 Flow Matching、GRPO、训练、LoRA、EMA 等。

In [1]:
import math
import copy
import random
import torch
import torch.nn as nn
import numpy as np
from collections import defaultdict

class Config:
    """所有超参数集中管理"""
    # --- Flow Matching / 采样 ---
    num_steps: int = 10            # 去噪步数 T
    noise_level: float = 0.5       # SDE 噪声强度，0=纯ODE
    sde_type: str = "sde"          # "sde" 或 "cps"
    guidance_scale: float = 7.0    # CFG 强度

    # --- Flow-GRPO-Fast：窗口化训练 ---
    sde_window_size: int = 3       # 仅在连续 3 个时间步注入 SDE
    sde_window_range: tuple = (0, 10)

    # --- GRPO ---
    group_size: int = 8            # 每个 prompt 生成多少张图
    num_prompts: int = 4           # 每轮训练多少个不同 prompt

    # --- 训练 ---
    learning_rate: float = 1e-4
    clip_range: float = 1e-4       # PPO 裁剪 ε
    adv_clip_max: float = 5.0      # 优势值裁剪上界
    beta: float = 5.0              # KL 正则化系数
    num_inner_epochs: int = 1      # 每批数据的重复训练次数

    # --- 其他 ---
    use_guard: bool = True         # GRPO-Guard 比率偏差校正
    ema_decay: float = 0.9         # EMA 衰减率
    lora_rank: int = 8             # LoRA 低秩维度
    lora_alpha: float = 16.0       # LoRA 缩放系数
    latent_dim: int = 4            # latent 空间维度
    hidden_dim: int = 64           # 隐藏层维度

cfg = Config()

## Toy Flow Matching 模型

学习一个速度场 $v_\theta(x_t, \sigma)$，将噪声 $x_1 \sim \mathcal{N}(0,I)$ 沿 ODE $\frac{dx}{dt} = v_\theta(x_t, t)$ 推向数据 $x_0$。

$\sigma=1$ 是纯噪声，$\sigma=0$ 是干净数据。在真实项目中，这就是 SD3 的 MMDiT 或 FLUX 的 Transformer。

In [2]:
class ToyVelocityModel(nn.Module):
    """Toy 速度场模型 v_θ(x_t, σ)，模拟 SD3/FLUX 的 Transformer
    
    输入: x_t (B,D) 当前 latent, sigma (B,) 当前噪声级别
    输出: v_θ (B,D) 预测的速度
    """
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        # 时间嵌入：将标量 sigma 编码为向量
        self.time_embed = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim),
        )
        # 速度预测网络（Tanh 限制输出范围，防止数值爆炸）
        self.net = nn.Sequential(
            nn.Linear(dim + hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, dim), nn.Tanh(),    # 速度输出在 [-1, 1]
        )

    def forward(self, x: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_embed(sigma.unsqueeze(-1))           # (B, hidden)
        return self.net(torch.cat([x, t_emb], dim=-1))         # (B, D)

## LoRA（Low-Rank Adaptation）—— 参数高效微调

不修改原始模型权重 $W$，而是学习低秩增量 $\Delta W = A \cdot B$，其中 $A \in \mathbb{R}^{d \times r}$, $B \in \mathbb{R}^{r \times d}$, $r \ll d$。

前向传播：$y = Wx + \frac{\alpha}{r} \cdot B(Ax)$

只训练 $A$ 和 $B$，冻结 $W$，大幅减少可训练参数（通常 < 5%）。初始化时 $B=0$ 确保 LoRA 增量为零，模型行为与原始一致。

In [3]:
class LoRALinear(nn.Module):
    """为 nn.Linear 添加 LoRA 适配器
    
    不修改原始权重 W，学习低秩增量 ΔW = A·B，其中 r ≪ d。
    前向: y = Wx + (α/r) · B(Ax)
    初始化: A 用 Kaiming，B 用零，确保初始时 LoRA 增量为 0。
    """
    def __init__(self, original: nn.Linear, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.original = original
        self.lora_A = nn.Linear(original.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, original.out_features, bias=False)
        self.scaling = alpha / rank

        nn.init.kaiming_uniform_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)    # 初始增量为 0

        for p in self.original.parameters():
            p.requires_grad = False    # 冻结原始权重

    def forward(self, x):
        return self.original(x) + self.lora_B(self.lora_A(x)) * self.scaling

def apply_lora(model: nn.Module, rank: int, alpha: float) -> list:
    """递归地将模型中所有 nn.Linear 替换为 LoRALinear"""
    trainable_params = []
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear):
            parts = name.split(".")
            parent = model
            for part in parts[:-1]:
                parent = getattr(parent, part)
            lora_module = LoRALinear(module, rank=rank, alpha=alpha)
            setattr(parent, parts[-1], lora_module)
            trainable_params.extend([lora_module.lora_A.weight, lora_module.lora_B.weight])
    return trainable_params

## Flow Matching Scheduler：sigma 调度与 ODE 步进

管理 sigma 序列（从 1→0），支持确定性 Euler ODE 步进：

$$x_{\sigma_{\text{prev}}} = x_\sigma + (\sigma_{\text{prev}} - \sigma) \cdot v_\theta(x_\sigma, \sigma)$$

ODE 是确定性的（相同初始噪声 → 相同结果）。为了 RL 的随机探索，后续会将 ODE 转换为 SDE。

In [4]:
def build_sigma_schedule(num_steps: int) -> torch.Tensor:
    """构建 sigma 调度表：从 1.0 (纯噪声) 到 0.0 (干净数据) 的递减序列"""
    sigmas = torch.linspace(1.0, 0.0, num_steps + 1)
    return sigmas

def get_sigma_max(sigmas: torch.Tensor) -> float:
    """获取 sigma_max = sigmas[1]，用于防止 SDE 步进中 sigma=1 时除零"""
    return sigmas[1].item()

def ode_step(v_theta: torch.Tensor, sigma: float, sigma_prev: float, x: torch.Tensor) -> torch.Tensor:
    """确定性 Euler ODE 步进（无随机性）
    
    公式: x_{σ_prev} = x_σ + (σ_prev − σ) · v_θ(x_σ, σ)
    dt < 0 因为 sigma 递减
    """
    dt = sigma_prev - sigma
    return x + dt * v_theta

## Anderson 反向 SDE 步进 + Log 概率计算

将确定性 ODE 转换为随机性 SDE，使同一 prompt 可以采样出不同图片。这是 **Flow-GRPO 的关键创新**：RL 需要随机策略来探索，纯 ODE 是确定性的。

**Anderson 反向 SDE 核心公式**：
- ODE: $dx = v \cdot dt$（确定性）
- SDE: $dx = [v + \text{noise correction}] \cdot dt + \tilde{\sigma} \cdot \sqrt{-dt} \cdot dW$

转移核 $p(x_{t-1}|x_t)$ 是高斯分布 $\mathcal{N}(\mu_\theta, \tilde{\sigma}^2 \cdot (-dt) \cdot I)$，因此 log prob 有 closed-form 表达式，且对 $\theta$ 可微。

**关键参数** `sigma_max`：当 $\sigma=1$ 时，$1-\sigma=0$ 导致除零，用 $\sigma_{\max} = \text{sigmas}[1]$ 替换避免溢出。

In [5]:
def sde_step_with_logprob(
    v_theta, sigma, sigma_prev, x,
    noise_level=0.7, sde_type="sde", sigma_max=0.9, prev_sample=None,
) -> tuple:
    """执行一步反向 SDE，并计算转移的 log 概率
    
    两种模式: "sde" (标准 Anderson 反向 SDE), "cps" (端点预测重参数化)
    """
    v_theta = v_theta.float()    # 确保 float32，避免 bf16 溢出
    x = x.float()
    if prev_sample is not None:
        prev_sample = prev_sample.float()

    dt = (sigma_prev - sigma).unsqueeze(-1)    # (B, 1), dt < 0
    sigma = sigma.unsqueeze(-1)
    sigma_prev = sigma_prev.unsqueeze(-1)

    if sde_type == "sde":
        # ---- σ̃_t: 噪声注入强度 ----
        # 当 sigma=1 时用 sigma_max 替换，避免 1/(1-1)=∞
        sigma_max_t = torch.full_like(sigma, sigma_max)
        sigma_safe = torch.where(sigma == 1, sigma_max_t, sigma)
        std_dev_t = torch.sqrt(sigma / (1 - sigma_safe)) * noise_level

        # ---- 修正后的转移均值 μ_θ ----
        prev_mean = (
            x * (1 + std_dev_t**2 / (2 * sigma_safe) * dt) +
            v_theta * (1 + std_dev_t**2 * (1 - sigma) / (2 * sigma_safe)) * dt
        )

        if prev_sample is None:
            # 采样阶段：从高斯转移核中采样
            noise = torch.randn_like(x)
            prev_sample = prev_mean + std_dev_t * torch.sqrt(-dt) * noise

        # ---- 高斯 Log 概率 ----
        variance = (std_dev_t * torch.sqrt(-dt)) ** 2
        log_prob = (
            -((prev_sample.detach() - prev_mean) ** 2) / (2 * variance + 1e-8)
            - torch.log(std_dev_t * torch.sqrt(-dt) + 1e-8)
            - 0.5 * math.log(2 * math.pi)
        )
        log_prob = log_prob.mean(dim=tuple(range(1, log_prob.ndim)))    # (B,)
        return prev_sample, log_prob, prev_mean, std_dev_t

    elif sde_type == "cps":
        return _sde_step_cps(v_theta, sigma, sigma_prev, dt, x, noise_level, prev_sample)

## CPS（Coefficients-Preserving Sampling）

CPS 是 Flow-GRPO 提出的另一种 SDE 参数化方式。不直接修正漂移项，而是从预测的端点重新参数化转移均值：

- $\hat{x}_0 = x - \sigma \cdot v_\theta$（预测的干净图像）
- $\hat{x}_1 = x + (1-\sigma) \cdot v_\theta$（预测的纯噪声）
- $\tilde{\sigma}_t = \sigma_{\text{prev}} \cdot \sin(\text{noise\_level} \cdot \pi/2)$
- $\mu_\theta = \hat{x}_0 \cdot (1-\sigma_{\text{prev}}) + \hat{x}_1 \cdot \sqrt{\sigma_{\text{prev}}^2 - \tilde{\sigma}_t^2}$

避免了标准 SDE 中 $\sigma=1$ 时的除零问题，经验上产生更高质量的样本。

In [6]:
def _sde_step_cps(v_theta, sigma, sigma_prev, dt, x, noise_level, prev_sample):
    """Coefficients-Preserving Sampling (sde_type='cps')
    
    与标准 SDE 的区别：
    - 标准 SDE 在漂移项上做修正（涉及 1/(1-σ)，sigma=1 时有数值问题）
    - CPS 从预测的端点 (x_0, x_1) 重新参数化均值，避免了这个问题
    """
    # σ̃_t = σ_prev · sin(noise_level · π/2)
    std_dev_t = sigma_prev * math.sin(noise_level * math.pi / 2)

    # 从速度场预测端点
    pred_x0 = x - sigma * v_theta          # 预测的干净数据 (sigma=0 端)
    pred_x1 = x + (1 - sigma) * v_theta    # 预测的纯噪声 (sigma=1 端)

    # 用端点重新参数化转移均值
    prev_mean = (
        pred_x0 * (1 - sigma_prev) +
        pred_x1 * torch.sqrt(torch.clamp(sigma_prev**2 - std_dev_t**2, min=0))
    )

    if prev_sample is None:
        noise = torch.randn_like(x)
        prev_sample = prev_mean + std_dev_t * noise

    # CPS 的 log prob（简化版，去掉常数项，不影响梯度方向）
    log_prob = -((prev_sample.detach() - prev_mean) ** 2)
    log_prob = log_prob.mean(dim=tuple(range(1, log_prob.ndim)))    # (B,)
    return prev_sample, log_prob, prev_mean, std_dev_t

## 带 Log Prob 的完整采样流程

从纯噪声 $x_1 \sim \mathcal{N}(0,I)$ 出发，逐步执行 SDE 反向步进，同时记录每步的中间 latent 和 log prob。

**Flow-GRPO-Fast（窗口化训练）**：仅在一个随机窗口内注入 SDE 噪声，窗口外使用确定性 ODE。只返回窗口内的轨迹，大幅加速训练（~5x），同时保持生成质量。

In [7]:
@torch.no_grad()
def sample_trajectories(
    model: nn.Module, sigmas: torch.Tensor,
    batch_size: int, latent_dim: int,
    noise_level: float = 0.7, sde_type: str = "sde", sigma_max: float = 0.9,
    sde_window_size: int = 0, sde_window_range: tuple = (0, 10),
) -> dict:
    """执行完整的去噪采样，记录轨迹
    
    Flow-GRPO-Fast: 如果 sde_window_size > 0，仅在一个随机窗口内注入 SDE 噪声，
    窗口外使用确定性 ODE。只返回窗口内的轨迹，大幅加速训练（~5x）。
    """
    num_steps = len(sigmas) - 1
    device = next(model.parameters()).device
    x = torch.randn(batch_size, latent_dim, device=device)    # 初始噪声 x_1 ~ N(0, I)

    # 确定 SDE 窗口（Flow-GRPO-Fast）
    if sde_window_size > 0:
        start = random.randint(sde_window_range[0], sde_window_range[1] - sde_window_size)
        sde_window = (start, start + sde_window_size)
    else:
        sde_window = (0, num_steps - 1)    # 标准版：所有步骤都用 SDE

    all_latents, all_log_probs, all_timesteps = [], [], []

    for i in range(num_steps):
        sigma_cur, sigma_next = sigmas[i], sigmas[i + 1]

        # 判断当前步是否在 SDE 窗口内
        if i < sde_window[0]:
            cur_noise_level = 0.0          # 窗口前：确定性 ODE
        elif i == sde_window[0]:
            cur_noise_level = noise_level  # 窗口起始：开始注入噪声
            all_latents.append(x)          # 记录初始 latent
        elif sde_window[0] < i < sde_window[1]:
            cur_noise_level = noise_level  # 窗口内：SDE
        else:
            cur_noise_level = 0.0          # 窗口后：确定性 ODE

        # 模型前向：预测速度 v_θ(x_t, σ_t)
        sigma_batch = sigma_cur.expand(batch_size).to(device)
        v_theta = model(x, sigma_batch)
        sigma_next_batch = sigma_next.expand(batch_size).to(device)

        if cur_noise_level > 0:
            # SDE 步进（有随机性，记录 log prob）
            x_next, log_prob, _, _ = sde_step_with_logprob(
                v_theta, sigma_batch, sigma_next_batch, x,
                noise_level=cur_noise_level, sde_type=sde_type, sigma_max=sigma_max,
            )
        else:
            # ODE 步进（确定性，不记录 log prob）
            x_next = ode_step(v_theta, sigma_cur.item(), sigma_next.item(), x)
            log_prob = torch.zeros(batch_size, device=device)

        # 仅记录窗口内的轨迹
        if sde_window[0] <= i < sde_window[1]:
            all_latents.append(x_next)
            all_log_probs.append(log_prob)
            all_timesteps.append(sigma_cur)
        x = x_next

    return {
        "latents": torch.stack(all_latents, dim=1),    # (B, W+1, D)
        "log_probs": torch.stack(all_log_probs, dim=1), # (B, W)
        "timesteps": torch.stack(all_timesteps),         # (W,)
        "images": x,                                     # (B, D) 最终结果
    }

## 奖励函数

RL 需要奖励信号来指导训练。在 Flow-GRPO 中，奖励来自预训练的评分模型（PickScore、CLIP Score、GenEval 等），支持多个奖励的加权组合。本 demo 用简单的二次函数模拟。

In [8]:
def toy_reward_fn(images: torch.Tensor, prompts: list) -> dict:
    """Toy 奖励函数：用简单的二次函数模拟奖励信号
    
    在真实项目中会调用 PickScore/CLIP/GenEval 等评分模型。
    返回多个奖励维度的加权组合。
    """
    # score_a: 鼓励 latent 接近目标值 0.5（模拟 PickScore）
    target = 0.5
    score_a = -((images - target) ** 2).mean(dim=-1)    # 越接近 0.5 分越高

    # score_b: 鼓励 latent 的方差小（模拟 CLIP Score）
    score_b = -images.var(dim=-1)

    # 加权平均（对应 multi-reward 的加权组合）
    avg = 0.6 * score_a + 0.4 * score_b
    return {"avg": avg, "score_a": score_a, "score_b": score_b}

## Per-Prompt 优势估计（GRPO 核心）

GRPO 的关键创新：用 group 内的均值/标准差作为 baseline，替代 PPO 中的 learned value function。

$$\text{advantage}_i = \frac{\text{reward}_i - \text{mean}(\text{group})}{\text{std}(\text{group})}$$

同一个 prompt 生成 $G$ 张图，构成一个 group。每张图的优势 = 自身 reward 相对 group 的标准化偏差。**无需额外的 critic 网络**，大幅简化训练。

支持 `global_std` 模式：用所有 prompt 的全局标准差，更加稳定。

In [9]:
class PerPromptStatTracker:
    """跟踪每个 prompt 的 reward 统计量，用于 GRPO 优势估计"""
    def __init__(self, global_std: bool = True):
        self.global_std = global_std
        self.stats = {}    # prompt → reward 历史列表

    def update(self, prompts: list, rewards: np.ndarray) -> np.ndarray:
        """更新统计量并计算 per-prompt 优势"""
        prompts = np.array(prompts)
        rewards = np.array(rewards, dtype=np.float64)
        unique = np.unique(prompts)
        advantages = np.zeros_like(rewards)

        # 第一步：收集每个 prompt 的历史 reward
        for prompt in unique:
            mask = prompts == prompt
            if prompt not in self.stats:
                self.stats[prompt] = []
            self.stats[prompt].extend(rewards[mask])

        # 第二步：计算 GRPO 优势 = (reward - mean) / std
        for prompt in unique:
            mask = prompts == prompt
            prompt_rewards = rewards[mask]
            mean = np.mean(self.stats[prompt], axis=0, keepdims=True)    # 该 prompt 的历史均值

            if self.global_std:
                std = np.std(rewards, axis=0, keepdims=True) + 1e-4      # 全局标准差（更稳定）
            else:
                std = np.std(self.stats[prompt], axis=0, keepdims=True) + 1e-4

            advantages[mask] = (prompt_rewards - mean) / std
        return advantages

    def clear(self):
        """每个 epoch 结束后清空统计量"""
        self.stats = {}

## 重新计算 Log Prob

训练时，对之前采样的轨迹中的每个转移 $(x_t \to x_{t-1})$，用**当前策略**（可能已更新）重新计算 log prob。

然后与采样时的 old log prob 做比值，得到 importance ratio：

$$\text{ratio} = \frac{\pi_\theta(x_{t-1}|x_t)}{\pi_{\text{old}}(x_{t-1}|x_t)} = \exp(\log p_\theta - \log p_{\text{old}})$$

这是 PPO/GRPO 的核心机制。

In [10]:
def compute_log_prob(
    model: nn.Module,
    x_t: torch.Tensor,            # (B, D) 当前步 latent
    sigma_cur: torch.Tensor,      # (B,) 当前 sigma
    sigma_next: torch.Tensor,     # (B,) 下一个 sigma
    x_next_stored: torch.Tensor,  # (B, D) 采样时保存的下一步 latent
    noise_level: float,
    sde_type: str = "sde",
    sigma_max: float = 0.9,
) -> tuple:
    """用当前模型重新计算转移 (x_t → x_{t-1}) 的 log prob
    
    关键：prev_sample 传入的是之前采样时保存的 x_{t-1}，
    函数不采样新的 x_{t-1}，而是计算当前策略对已有转移的概率。
    这样才能计算 importance ratio = π_θ / π_old。
    """
    # 用当前模型预测速度
    v_theta = model(x_t, sigma_cur)

    # 用保存的 x_{t-1} 计算 log prob（不重新采样）
    _, log_prob, prev_mean, std_dev_t = sde_step_with_logprob(
        v_theta, sigma_cur, sigma_next, x_t,
        noise_level=noise_level, sde_type=sde_type,
        sigma_max=sigma_max,
        prev_sample=x_next_stored,    # 关键：使用保存的样本
    )
    return log_prob, prev_mean, std_dev_t

## GRPO 损失函数

### PPO-style 裁剪代理损失

$$\mathcal{L}_{\text{clipped}} = \max(-\text{adv} \cdot \text{ratio},\ -\text{adv} \cdot \text{clamp}(\text{ratio}, 1-\varepsilon, 1+\varepsilon))$$

其中 $\text{ratio} = \exp(\log \pi_\theta - \log \pi_{\text{old}})$ 是重要性采样比率。取 max 确保裁剪生效：对正优势限制 ratio 不要太大，对负优势限制 ratio 不要太小。

### KL 散度正则化

$$\mathcal{L}_{\text{KL}} = \beta \cdot \frac{\|\mu_\theta - \mu_{\text{ref}}\|^2}{2\sigma^2}$$

防止策略偏离 reference model 太远，保持生成质量。

In [11]:
def grpo_policy_loss(
    log_prob: torch.Tensor,       # (B,) 当前策略的 log prob
    old_log_prob: torch.Tensor,   # (B,) 采样时的 log prob
    advantages: torch.Tensor,     # (B,) 优势值
    clip_range: float = 1e-4,     # PPO 裁剪 ε
    adv_clip_max: float = 5.0,    # 优势裁剪上界
) -> tuple:
    """计算 PPO-style 裁剪代理损失"""
    # 裁剪优势值，防止极端优势导致梯度爆炸
    advantages = torch.clamp(advantages, -adv_clip_max, adv_clip_max)

    # 重要性采样比率: ratio = π_θ / π_old = exp(log_prob - old_log_prob)
    ratio = torch.exp(log_prob - old_log_prob)

    # PPO 裁剪代理损失:
    # unclipped = -advantage * ratio (无裁剪的策略梯度)
    # clipped   = -advantage * clamp(ratio, 1-ε, 1+ε) (裁剪后的版本)
    # 取 max 确保裁剪生效
    unclipped_loss = -advantages * ratio
    clipped_loss = -advantages * torch.clamp(ratio, 1.0 - clip_range, 1.0 + clip_range)
    policy_loss = torch.mean(torch.maximum(unclipped_loss, clipped_loss))

    clip_fraction = torch.mean((torch.abs(ratio - 1.0) > clip_range).float())
    return policy_loss, ratio, clip_fraction


def compute_kl_loss(
    prev_mean: torch.Tensor,       # (B, D) 当前策略的转移均值 μ_θ
    prev_mean_ref: torch.Tensor,   # (B, D) reference model 的转移均值 μ_ref
    std_dev_t: torch.Tensor,       # 转移标准差 σ̃
    beta: float = 0.04,            # KL 系数
) -> torch.Tensor:
    """KL 散度正则化损失
    
    当两个策略的转移核都是高斯 N(μ, σ²I) 且 σ 相同时：
    KL(π_θ || π_ref) = ||μ_θ - μ_ref||² / (2σ²)
    """
    sigma_sq = std_dev_t.mean() ** 2 + 1e-8    # 加小数防止除零
    kl = ((prev_mean - prev_mean_ref) ** 2).mean() / (2 * sigma_sq)
    return beta * kl

## GRPO-Guard：比率偏差校正

Flow-GRPO 发现 importance ratio 存在固有偏差：ratio 的均值系统性地 < 1，尤其在低噪声时间步，导致 PPO 的 clipping 对正样本 clip 不足 → 梯度爆炸。

**GRPO-Guard 的三重修正**：
1. **偏差校正**：$\text{bias} = \|\mu_\theta - \mu_{\text{stored}}\|^2 / (2\sigma^2)$，补偿 ratio 系统性 < 1
2. **时间步缩放**：用 $\sigma_t \cdot \sqrt{-dt}$ 缩放 log-ratio，使不同时间步的梯度量级一致
3. **梯度重加权**：损失除以 $(-dt)$，进一步平衡时间步间的梯度

In [12]:
def grpo_guard_ratio(
    log_prob: torch.Tensor,         # 当前策略 log prob
    old_log_prob: torch.Tensor,     # 采样时 log prob
    prev_mean: torch.Tensor,        # 当前策略转移均值 μ_θ
    stored_mean: torch.Tensor,      # 采样时转移均值（保存的）
    std_dev_t: torch.Tensor,        # 转移标准差 σ̃
    sqrt_dt: torch.Tensor,          # √(-dt)
) -> torch.Tensor:
    """GRPO-Guard 修正后的 importance ratio
    
    与标准 GRPO 的区别：
    - 标准: ratio = exp(log_prob - old_log_prob)
    - Guard: ratio = exp((log_prob - old_log_prob + bias_correction) * scaling)
    """
    sigma_t = std_dev_t.mean()
    sigma_sq = sigma_t ** 2 + 1e-8

    # 估计 ratio 的偏差（偏差来自均值漂移）
    # bias = ||μ_θ - μ_stored||² / (2σ²)
    ratio_mean_bias = ((prev_mean - stored_mean) ** 2).mean() / (2 * sigma_sq)

    # 修正后的 log-ratio：加上偏差校正，用 σ_t·√(-dt) 缩放
    corrected_log_ratio = (log_prob - old_log_prob + ratio_mean_bias) * (sqrt_dt * sigma_t)

    ratio = torch.exp(corrected_log_ratio)
    return ratio

## EMA（指数移动平均）

维护模型参数的滑动平均：$\theta_{\text{ema}} = \eta \cdot \theta_{\text{ema}} + (1-\eta) \cdot \theta$

EMA 参数更平滑，通常比最终参数有更好的生成质量。包含 warmup 机制：训练初期用小衰减快速跟踪，逐步过渡到目标衰减。

In [13]:
class SimpleEMA:
    """简化版 EMA（指数移动平均），保留核心逻辑
    
    EMA 维护模型参数的滑动平均，用于评估和保存。
    θ_ema = decay · θ_ema + (1 - decay) · θ_current
    EMA 参数更平滑，通常比最终参数有更好的生成质量。
    """
    def __init__(self, parameters, decay: float = 0.9):
        self.decay = decay
        self.ema_params = [p.clone().detach() for p in parameters]    # 保存参数副本

    def get_current_decay(self, step: int) -> float:
        """warmup 衰减率：初始时用小衰减快速跟踪，逐步过渡到目标衰减"""
        return min((1 + step) / (10 + step), self.decay)

    @torch.no_grad()
    def step(self, parameters, step: int):
        """更新 EMA 参数: θ_ema += (1-decay) * (θ - θ_ema)"""
        one_minus_decay = 1 - self.get_current_decay(step)
        for ema_p, p in zip(self.ema_params, parameters):
            if p.requires_grad:
                ema_p.add_(one_minus_decay * (p.data - ema_p))

    def copy_to(self, parameters):
        """将 EMA 参数复制到模型（用于评估）"""
        for ema_p, p in zip(self.ema_params, parameters):
            p.data.copy_(ema_p)

In [14]:
torch.manual_seed(42)    # 固定随机种子
device = "cpu"
print("=" * 70)
print("  Flow-GRPO 最简 Demo")
print("=" * 70)

  Flow-GRPO 最简 Demo


## 完整训练循环

Flow-GRPO 的训练分 **三个阶段**，每个 epoch 重复：

| 阶段 | 名称 | 核心操作 |
|------|------|---------|
| Phase 1 | 采样 (Sampling) | 用当前策略采样 G 张图，记录完整去噪轨迹 |
| Phase 2 | 优势估计 (Advantage) | Per-Prompt 组内归一化，无需 critic |
| Phase 3 | 策略更新 (Training) | PPO 裁剪损失 + KL 正则化更新 LoRA 参数 |

### 初始化

创建模型、reference model、LoRA 适配器、优化器、EMA 等组件。

In [15]:
# ---- 创建模型 ----
model = ToyVelocityModel(dim=cfg.latent_dim, hidden_dim=cfg.hidden_dim).to(device)
print(f"\n[模型] 原始参数量: {sum(p.numel() for p in model.parameters()):,}")

# ---- 创建 reference model（KL 正则化用）----
# 必须在应用 LoRA 之前创建，保存的是无 LoRA 的原始模型
ref_model = copy.deepcopy(model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

# ---- 应用 LoRA ----
trainable_params = apply_lora(model, rank=cfg.lora_rank, alpha=cfg.lora_alpha)
num_trainable = sum(p.numel() for p in trainable_params)
num_total = sum(p.numel() for p in model.parameters())
print(f"[LoRA] 可训练参数: {num_trainable:,} / {num_total:,} ({100*num_trainable/num_total:.1f}%)")

# ---- 优化器（只优化 LoRA 参数）----
optimizer = torch.optim.AdamW(trainable_params, lr=cfg.learning_rate)

# ---- EMA ----
ema = SimpleEMA(trainable_params, decay=cfg.ema_decay)

# ---- 优势估计跟踪器 ----
stat_tracker = PerPromptStatTracker(global_std=True)

# ---- sigma 调度表 ----
sigmas = build_sigma_schedule(cfg.num_steps)
sigma_max = get_sigma_max(sigmas)
print(f"\n[调度] sigma: {[f'{s:.2f}' for s in sigmas.tolist()]}")
print(f"[调度] sigma_max = {sigma_max:.2f}")

# ---- 模拟 prompt 池 ----
prompts_pool = [
    "a cat sitting on a mat",
    "a dog running in a park",
    "a bird flying in the sky",
    "a fish swimming in the sea",
]

global_step = 0
for epoch in range(3):    # 3 个 epoch 演示
    print(f"\n{'='*50}\n  Epoch {epoch}\n{'='*50}")


[模型] 原始参数量: 8,964
[LoRA] 可训练参数: 3,144 / 12,108 (26.0%)

[调度] sigma: ['1.00', '0.90', '0.80', '0.70', '0.60', '0.50', '0.40', '0.30', '0.20', '0.10', '0.00']
[调度] sigma_max = 0.90

  Epoch 0

  Epoch 1

  Epoch 2


### Phase 1：采样轨迹（Sampling）

对每个 prompt，用当前策略采样 $G$ 张图（GRPO group），记录完整的去噪轨迹：
- **中间状态** `latents`：$(x_1, x_{\sigma_1}, ..., x_0)$，所有时间步的 latent
- **log prob**：每步 SDE 转移的对数概率 $\log p(x_{t-1}|x_t)$

使用 SDE 采样（注入噪声）使得同一 prompt 可以生成不同的图片，这是 RL 探索的基础。

In [16]:
# ============================================================
# Phase 1: SAMPLING — 采样轨迹
# ============================================================
print("\n📦 Phase 1: 采样轨迹...")
model.eval()

all_samples = []
for batch_idx in range(cfg.num_prompts):
    # 选一个 prompt，重复 group_size 次（GRPO group）
    prompt = prompts_pool[batch_idx % len(prompts_pool)]
    batch_prompts = [prompt] * cfg.group_size

    # 采样：生成 group_size 张图，记录完整轨迹
    trajectories = sample_trajectories(
        model, sigmas,
        batch_size=cfg.group_size,
        latent_dim=cfg.latent_dim,
        noise_level=cfg.noise_level,
        sde_type=cfg.sde_type,
        sigma_max=sigma_max,
        sde_window_size=cfg.sde_window_size,
        sde_window_range=cfg.sde_window_range,
    )

    # 计算奖励
    rewards_dict = toy_reward_fn(trajectories["images"], batch_prompts)

    # 保存轨迹数据
    all_samples.append({
        "prompts": batch_prompts,
        "latents": trajectories["latents"],        # (G, T_win+1, D)
        "log_probs": trajectories["log_probs"],     # (G, T_win)
        "timesteps": trajectories["timesteps"],     # (T_win,)
        "rewards": rewards_dict["avg"],             # (G,)
    })
    print(f"  Prompt: '{prompt}' → reward: mean={rewards_dict['avg'].mean():.4f}")

# 合并所有 batch 的样本
all_latents = torch.cat([s["latents"] for s in all_samples], dim=0)
all_log_probs = torch.cat([s["log_probs"] for s in all_samples], dim=0)
all_timesteps = all_samples[0]["timesteps"]
all_rewards = torch.cat([s["rewards"] for s in all_samples], dim=0)
all_prompts = [p for s in all_samples for p in s["prompts"]]

# latents[:, :-1] 是 x_t, latents[:, 1:] 是 x_{t-1}
x_t_all = all_latents[:, :-1]       # (N, T_win, D)
x_next_all = all_latents[:, 1:]     # (N, T_win, D)
num_train_timesteps = all_timesteps.shape[0]
total_batch = x_t_all.shape[0]

print(f"\n  总样本数: {total_batch}, 训练时间步数: {num_train_timesteps}")


📦 Phase 1: 采样轨迹...
  Prompt: 'a cat sitting on a mat' → reward: mean=-1.1436
  Prompt: 'a dog running in a park' → reward: mean=-0.8818
  Prompt: 'a bird flying in the sky' → reward: mean=-1.4879
  Prompt: 'a fish swimming in the sea' → reward: mean=-1.2000

  总样本数: 32, 训练时间步数: 3


### Phase 2：计算 Per-Prompt 优势（Advantage）

GRPO 的核心创新：用 group 内的统计量替代 learned value function。

$$\text{advantage}_i = \frac{\text{reward}_i - \text{mean}(\text{group})}{\text{std}(\text{group})}$$

- 同一个 prompt 生成 $G$ 张图，构成一个 group
- 每张图的优势 = 自身 reward 相对 group 的标准化偏差
- 优势 > 0 的样本会被鼓励，< 0 的会被抑制
- 无需额外的 critic 网络，大幅简化训练

In [17]:
# ============================================================
# Phase 2: ADVANTAGE — 计算 per-prompt 优势
# ============================================================
print("\n📊 Phase 2: 计算 GRPO 优势...")

# 将 reward 复制 T 份（每个时间步共享同一个优势值）
rewards_expanded = all_rewards.unsqueeze(1).repeat(1, num_train_timesteps)

# 通过 PerPromptStatTracker 计算 per-prompt 归一化优势
advantages = stat_tracker.update(all_prompts, rewards_expanded.numpy())
advantages = torch.tensor(advantages, dtype=torch.float32)

# 过滤零优势样本（group 内所有 reward 相同 → std=0 → 无学习信号）
nonzero_mask = advantages.abs().sum(dim=1) != 0
valid_count = nonzero_mask.sum().item()
print(f"  有效样本: {valid_count}/{len(all_prompts)} "
      f"(过滤 {len(all_prompts) - valid_count} 个零优势样本)")

if valid_count == 0:
    print("  ⚠️ 所有样本优势为零，跳过训练")
    stat_tracker.clear()
    # continue  # 在实际循环中会跳过

# 只保留有效样本
x_t_all = x_t_all[nonzero_mask]
x_next_all = x_next_all[nonzero_mask]
all_log_probs = all_log_probs[nonzero_mask]
advantages = advantages[nonzero_mask]

print(f"  优势范围: [{advantages.min():.3f}, {advantages.max():.3f}], "
      f"均值: {advantages.mean():.3f}")


📊 Phase 2: 计算 GRPO 优势...
  有效样本: 32/32 (过滤 0 个零优势样本)
  优势范围: [-3.257, 1.713], 均值: -0.000


### Phase 3：策略更新（Training）

对轨迹中的每个时间步，用当前策略重新计算 log prob，然后计算 GRPO 损失并更新参数：

1. **重新计算 log prob**：用当前 $v_\theta$ 对保存的转移 $(x_t \to x_{t-1})$ 重新计算概率
2. **计算 importance ratio**：$\text{ratio} = \exp(\log \pi_\theta - \log \pi_{\text{old}})$
3. **PPO 裁剪损失**：$\max(-\text{adv} \cdot \text{ratio}, -\text{adv} \cdot \text{clamp}(\text{ratio}))$
4. **KL 正则化**：$\beta \cdot \|\mu_\theta - \mu_{\text{ref}}\|^2 / (2\sigma^2)$
5. **反向传播 + 梯度裁剪 + 参数更新 + EMA 更新**

In [18]:
for inner_epoch in range(cfg.num_inner_epochs):
    # 随机打乱样本顺序
    perm = torch.randperm(total_batch)
    x_t_perm = x_t_all[perm]
    x_next_perm = x_next_all[perm]
    log_probs_perm = all_log_probs[perm]
    adv_perm = advantages[perm]

    model.train()
    metrics = defaultdict(list)

    for j in range(num_train_timesteps):
        # 当前时间步和下一个时间步的 sigma
        sigma_cur = all_timesteps[j].expand(total_batch).to(device)
        sigma_idx = (sigmas == all_timesteps[j]).nonzero(as_tuple=True)[0]
        if len(sigma_idx) > 0 and sigma_idx[0].item() + 1 < len(sigmas):
            sigma_next_val = sigmas[sigma_idx[0].item() + 1]
        else:
            sigma_next_val = torch.tensor(0.0)
        sigma_next = sigma_next_val.expand(total_batch).to(device)

        # ---- 用当前策略重新计算 log prob ----
        log_prob, prev_mean, std_dev_t = compute_log_prob(
            model, x_t=x_t_perm[:, j], sigma_cur=sigma_cur,
            sigma_next=sigma_next, x_next_stored=x_next_perm[:, j],
            noise_level=cfg.noise_level, sde_type=cfg.sde_type,
            sigma_max=sigma_max,
        )

        # ---- Reference model 的转移均值（KL 正则化用）----
        if cfg.beta > 0:
            with torch.no_grad():
                _, prev_mean_ref, _ = compute_log_prob(
                    ref_model, x_t=x_t_perm[:, j], sigma_cur=sigma_cur,
                    sigma_next=sigma_next, x_next_stored=x_next_perm[:, j],
                    noise_level=cfg.noise_level, sde_type=cfg.sde_type,
                    sigma_max=sigma_max,
                )

        adv_j = adv_perm[:, j]

        if cfg.use_guard:
            # ---- GRPO-Guard: 修正比率偏差 ----
            with torch.no_grad():
                _, stored_mean, _ = compute_log_prob(
                    model, x_t=x_t_perm[:, j], sigma_cur=sigma_cur,
                    sigma_next=sigma_next, x_next_stored=x_next_perm[:, j],
                    noise_level=cfg.noise_level, sde_type=cfg.sde_type,
                    sigma_max=sigma_max,
                )
            dt_val = sigma_next_val - all_timesteps[j]
            sqrt_dt = torch.sqrt(torch.tensor(abs(dt_val.item())))
            # 用修正后的 ratio 计算 PPO 损失
            ratio = grpo_guard_ratio(
                log_prob, log_probs_perm[:, j],
                prev_mean, stored_mean, std_dev_t, sqrt_dt,
            )
            adv_clipped = torch.clamp(adv_j, -cfg.adv_clip_max, cfg.adv_clip_max)
            unclipped = -adv_clipped * ratio
            clipped = -adv_clipped * torch.clamp(ratio, 1-cfg.clip_range, 1+cfg.clip_range)
            policy_loss = torch.mean(torch.maximum(unclipped, clipped))
            policy_loss = policy_loss / (sqrt_dt ** 2 + 1e-8)    # 梯度重加权
            clip_frac = torch.tensor(0.0)
        else:
            # ---- 标准 GRPO 损失 ----
            policy_loss, ratio, clip_frac = grpo_policy_loss(
                log_prob, log_probs_perm[:, j], adv_j,
                clip_range=cfg.clip_range, adv_clip_max=cfg.adv_clip_max,
            )

        # ---- 总损失 = 策略损失 + KL 正则化 ----
        loss = policy_loss
        if cfg.beta > 0:
            kl = compute_kl_loss(prev_mean, prev_mean_ref, std_dev_t, beta=cfg.beta)
            loss = loss + kl
            metrics["kl_loss"].append(kl)

        # ---- 反向传播 ----
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)  # 梯度裁剪
        optimizer.step()

        # ---- 记录指标 & EMA 更新 ----
        metrics["policy_loss"].append(policy_loss)
        metrics["loss"].append(loss)
        approx_kl = 0.5 * torch.mean((log_prob - log_probs_perm[:, j]) ** 2)
        metrics["approx_kl"].append(approx_kl)
        metrics["clip_frac"].append(clip_frac)
        ema.step(trainable_params, step=global_step)
        global_step += 1

    # 打印训练指标
    avg_metrics = {k: torch.stack(v).mean().item() for k, v in metrics.items()}
    print(f"  Inner epoch {inner_epoch}:")
    print(f"    policy_loss = {avg_metrics['policy_loss']:.6f}")
    print(f"    loss        = {avg_metrics['loss']:.6f}")
    print(f"    approx_kl   = {avg_metrics['approx_kl']:.6f}")
    if "kl_loss" in avg_metrics:
        print(f"    kl_loss     = {avg_metrics['kl_loss']:.6f}")
    print(f"    clip_frac   = {avg_metrics['clip_frac']:.4f}")

stat_tracker.clear()    # 清空统计量，准备下一个 epoch

  Inner epoch 0:
    policy_loss = 0.199932
    loss        = 0.199932
    approx_kl   = 0.036618
    kl_loss     = 0.000000
    clip_frac   = 0.0000


### 训练完成 & EMA 评估

训练结束后，将 EMA 参数复制回模型进行评估。评估时使用纯 ODE 采样（`noise_level=0`，确定性），不需要随机性。

In [19]:
print("\n" + "=" * 70)
print("  训练完成！")
print("=" * 70)

# ---- 用 EMA 参数评估 ----
print("\n📐 用 EMA 参数评估...")
ema.copy_to(trainable_params)    # 将 EMA 参数复制回模型
model.eval()
with torch.no_grad():
    eval_result = sample_trajectories(
        model, sigmas, batch_size=4, latent_dim=cfg.latent_dim,
        noise_level=0.0,     # 评估时用纯 ODE（确定性采样）
        sigma_max=sigma_max,
    )
    eval_rewards = toy_reward_fn(eval_result["images"], ["eval"] * 4)
    print(f"  评估奖励: mean={eval_rewards['avg'].mean():.4f}")


  训练完成！

📐 用 EMA 参数评估...
  评估奖励: mean=-1.4353
